# Scraping free patterns from Herrschners

### Setting the path for the image data and importing packages 

In [37]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

target_dir = "../frontend/src/assets/images"
os.makedirs(target_dir, exist_ok=True)

### Scraping every product link

In [38]:
def scrape_product(product_link, target_dir = "../frontend/src/assets/images"):
    row = {}
    os.makedirs(target_dir, exist_ok=True)
    
    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row

    # Information
    title_element = soup.find("h1", class_="productView-title")
    if title_element:
        raw_title = title_element.get_text(strip=True)
        clean_title = re.sub(r'\(?Free Download\)?', '', raw_title, flags=re.IGNORECASE).strip()
        row["title"] = clean_title
    else:
        row["title"] = ""
    names = soup.find_all(class_="productView-info-name")
    values = soup.find_all(class_="productView-info-value")
    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
        row[key] = value.get_text(strip=True)
    
    row["merged_info"] = " ".join(v.get_text(strip=True) for v in values)

    # Description
    desc_container = soup.find(class_="productView-desc-content")
    if desc_container:
        desciption = desc_container.get_text(" ", strip=True)
        row['description'] = desciption.split("Skill Level")[0].strip()
        skill_label = desc_container.find("b", string=re.compile("Skill Level", re.IGNORECASE))
    
        if skill_label and skill_label.next_sibling:
            row['skill_level'] = skill_label.next_sibling.strip()
        else:
            row['skill_level'] = ""

    # Review Count
    review_link = soup.find(class_="productView-reviewLink")
    if review_link:
        match = re.search(r"\d+", review_link.get_text())
        row["review_count"] = int(match.group()) if match else 0
    else:
        row["review_count"] = 0

    # Star Rating
    rating_div = soup.find(class_="productView-rating")
    if rating_div:
        stars = rating_div.find_all(class_=re.compile(r"icon--ratingFull"))
        row["star_rating"] = len(stars)
    else:
        row["star_rating"] = 0

    # Image Extraction and Download
    img_container = soup.find("span", class_="productView-imageCarousel-main-item-img-container")
    img_tag = img_container.find("img") if img_container else None
    
    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(target_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = f"{safe_filename}.jpg"
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")

    return row

In [39]:
base_url = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"
rows = []
page = 1

while True:
    print(f"Scraping page {page}...")
    url = base_url if page == 1 else f"{base_url}?page={page}"

    #if page == 2:
    #    break

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("article.card")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        product_link = card.get("data-href")
        row = {"product_link": product_link}

        if product_link:
            features = scrape_product(product_link)
            row.update(features)

        rows.append(row)
        time.sleep(0.5)

    page += 1

df = pd.DataFrame(rows)

df.columns = [col.replace(':', '') for col in df.columns]
print("Scraping complete!")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
No more products found. Ending scrape.
Scraping complete!


In [62]:
df.head()

,product_link,title,sku,upc,project_type,skill,difficulty,merged_info,description,skill_level,review_count,star_rating,image_url,local_path,occasion,season,theme,fiber_content,yarn_weight,thread_size
0,https://herrschners.com/herrschners-modern-mes...,Modern Mesa Pillow,F01731,,Pillow,Knit,Beginner,F01731 Pillow Knit Easy,Create a cozy sitting area filled with hand-st...,beginner,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,Herrschners_Modern_Mesa_Pillow.jpg,NaN,NaN,NaN,NaN,NaN,NaN
1,https://herrschners.com/herrschners-sunrise-st...,Sunrise Stroll Tote,F01730,,Tote,Crochet,Beginner,F01730 Tote Crochet Easy,Accessorize in style with this textured two-to...,beginner,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,Herrschners_Sunrise_Stroll_Tote.jpg,NaN,NaN,NaN,NaN,NaN,NaN
2,https://herrschners.com/village-yarn-blue-burs...,Village Yarn Blue Burst Pot Holders,F01729,,Pot Holders,Crochet,Beginner,F01729 Pot Holders Crochet Easy,Brighten your kitchen with a splash of color! ...,beginner,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,Village_Yarn_Blue_Burst_Pot_Holders.jpg,NaN,NaN,NaN,NaN,NaN,NaN
3,https://herrschners.com/willow-yarns-garden-so...,Willow Yarns Garden Song Shawlette,F01728,,Shawl,Knit,Beginner,F01728 Shawl Knit Easy,Be ready for backyard garden parties with this...,beginner,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,Willow_Yarns_Garden_Song_Shawlette.jpg,NaN,NaN,NaN,NaN,NaN,NaN
4,https://herrschners.com/herrschners-english-ga...,English Garden Doily,F01727,,Doily,Crochet,Beginner,F01727 Doily Crochet Easy,No green thumb is needed to grow a garden of t...,beginner,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,Herrschners_English_Garden_Doily.jpg,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
df['skill_level'].value_counts()

skill_level
Easy                 240
Intermediate         141
                      60
Beginner              19
Experienced            3
Easy crochet           3
Easy knit              2
Advanced               2
Intermedate            1
Intermediate Knit      1
Easy Knit              1
Beginner crochet       1
Name: count, dtype: int64

In [57]:
import numpy as np
def clean_skill(x):
    if pd.isna(x):
        return np.nan
    
    x = x.lower()
    
    if "beginner" in x:
        return "beginner"
    elif "intermediate" in x:
        return "intermediate"
    elif "advanced" in x or "experienced" in x:
        return "advanced"
    elif "easy" in x:
        return "beginner"
    else:
        return np.nan

df["skill_level"] = df["skill_level"].apply(clean_skill)

In [58]:
df['skill_level'].value_counts()

skill_level
beginner        266
intermediate    142
advanced          5
Name: count, dtype: int64

In [59]:
df['title'] = df['title'].str.replace("Herrschners", "", regex=False).str.strip()

In [60]:
df['product_link'].nunique()

474

In [61]:
df.to_csv("../data/herrschners_crochet_patterns.csv", index=False)